# HypeCheck: do headline-style posts predict memecoin direction?
**Veritas AI International High School Hackathon 2026 · AI + Finance**

**Team:** Nathaniel Wong (model), Aaden Yick (data), Alastair Chu (demo and write-up) · Malvern College Hong Kong.

**Pitch:** HypeCheck helps a beginner crypto investor test whether bullish social posts deserve forecasting trust, by comparing sentiment-based next-day forecasts against simple rules and actual market outcomes.

**How to run:** Runtime → Run all. No keys, uploads or live Binance API calls. The default uses verified cached outputs from a pinned pretrained sentiment model and trains direction classifiers afresh. A recompute switch reruns frozen sentiment inference. The encoder is not fine-tuned.

This expands the original study to **13 memecoins**. There is no 40-tweet test and no 15/25 sentiment quota: all eligible naturally occurring posts are retained. X posts are our headline-style source, not newspaper headlines or a complete X feed.

The section sequence follows the sample: load → inspect → split → baseline → first model → mistakes → later-data test → measured improvement → demo → results → limitations and sources. New experiment, same archive: this is exploratory, not independent confirmation.

*This project is a prototype and is not financial advice.*

In [1]:
import sys,subprocess,json,hashlib,urllib.request
from pathlib import Path
IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    pins=['numpy==2.5.4','pandas==3.1.1','pyarrow==26.0.1','scikit-learn==1.9.1','requests==2.34.2','transformers==5.23.0','safetensors==0.8.0']
    subprocess.run([sys.executable,'-m','pip','install','-q',*pins],check=True)
for name in ['data','docs','artifacts']:Path(name).mkdir(exist_ok=True)
print('Runtime:',sys.version,'Colab:',IN_COLAB)
print('The default direction classifiers are trained during this run.')

Runtime: 3.12.14 (main, Sep 24 2026, 17:57:43) [Clang 22.1.3 ] Colab: False
The default direction classifiers are trained during this run.


## 1. Load the data
Load archived posts, real daily prices and pretrained sentiment scores. The short loader fetches a **fixed version** from GitHub and verifies every file with SHA-256. No API keys.

[Setup code](https://github.com/adeviwon/hypecheck/blob/main/colab_setup.py) · [Model code](https://github.com/adeviwon/hypecheck/blob/main/headline_study.py)

In [2]:
setup_url='https://raw.githubusercontent.com/adeviwon/hypecheck/76b18f042c32b36f4eff541962680d14671ebda2/colab_setup.py'
payload=urllib.request.urlopen(setup_url,timeout=30).read()
if hashlib.sha256(payload).hexdigest()!='48e912fec0f69329b2ed93ef5da6cca4890ec191fa6e84367eece7be2a246cb4':raise ValueError("Setup file changed")
Path("colab_setup.py").write_bytes(payload)
from colab_setup import prepare
RECOMPUTE_SENTIMENT=False
prepare(recompute_sentiment=RECOMPUTE_SENTIMENT)
import pipeline as core
from expanded_inputs import select_posts,load,split_data,COINS
raw,_=core.load_posts()
print(f"Loaded {len(raw):,} archived posts.")

Verified source and market data. Sentiment: verified cache
Loaded 104,379 archived posts.


## 2. Look at the data before training
Keep original posts, bounded nonempty text, exact supported ticker tags and earliest exact-text duplicates. The later of publication and archive collection time is conservative availability. A post from day d feeds only the forecast for day d+1.

Sentiment classes are outputs of the pretrained model, not human labels. Bullish does not mean the coin actually rose. Multi-coin posts can contribute to multiple coin-days; tweet count, market-bar count and prediction-row count are different units.

In [3]:
from expanded_inputs import inspect_posts
selected=inspect_posts(raw)
import pandas as pd,numpy as np

Eligible original posts: 1773
Coins: DOGE, SHIB, PEPE, BONK, FLOKI, WIF, TRUMP, NEIRO, PENGU, PNUT, TURBO, BOME, MEME
                 available_at                                                                                                                                                          text
2024-03-22 04:49:17.553+00:00                                                                                                                         Take partial profit fam 🔥 $bome #bome
2024-03-22 06:29:03.335+00:00                                                                                                              $doge legit best looking major by a mile atm imo
2024-03-22 07:19:03.469+00:00 The doge of this cycle should go for the next round anytime now IMO.\n\nAfter consolidation comes continuation.\nInvalidation below the horizontal.\n\n$FLOKI
2024-03-22 10:29:17.676+00:00                                               Bagged some \n\nBeing honest here. I think $doge gonna

## 3. Split into training, validation and later-date test data
**Never split these financial rows randomly.** Use shared chronological boundaries, approximately 60/20/20 across UTC dates, and purge the adjacent day at each boundary. Earlier target windows must end before the next partition begins.

The label is next-UTC-day **close > open**; flat counts as not up. Only strictly prior-day sentiment and lagged market features can be predictors. Scaling and coefficients are fitted on training rows; regularisation is selected by validation Brier error. The expanded universe and period were not a pristine new preregistration; we disclose archive reuse.

In [4]:
raw,scored,prices,frame=load()
print('Natural sentiment counts:',scored[['bullish','bearish','neutral']].idxmax(axis=1).value_counts().to_dict())
train,validation,test=split_data(frame)
print(pd.DataFrame([{'partition':name,'first':str(x.date.min().date()),'last':str(x.date.max().date()),'coin_days':len(x),'UTC_dates':x.date.nunique()} for name,x in [('train',train),('validation',validation),('test',test)]]).to_string(index=False))
assert train.date.max()+pd.Timedelta(days=1)<validation.date.min()
assert validation.date.max()+pd.Timedelta(days=1)<test.date.min()
assert not set(train.date)&set(test.date)
print('Boundary and shared-date leakage checks passed.')

Natural sentiment counts: {'bullish': 1238, 'bearish': 358, 'neutral': 177}
 partition      first       last  coin_days  UTC_dates
     train 2024-03-01 2025-09-29       6171        578
validation 2025-10-01 2026-04-10       2496        192
      test 2026-04-12 2026-10-21       2509        193
Boundary and shared-date leakage checks passed.


## 4. Baselines: always up and previous-day direction
Always up ignores all text. Persistence copies the previous calendar UTC day's **open-to-close** direction, matching the target—not previous close-to-close return. We also report training majority and a training-prevalence probability baseline so a falling market cannot masquerade as text skill.

In [5]:
from headline_study import previous_day
persistence=np.full(len(test),np.nan)
for coin in COINS:
    mask=test.coin.eq(coin).to_numpy()
    persistence[mask]=test.loc[mask,'date'].map(previous_day(prices[coin])).to_numpy()
assert np.isfinite(persistence).all()
y=test.actual_up.to_numpy()
baselines={'Always up':np.ones(len(test)),'Previous-day direction':persistence,'Training majority':np.full(len(test),float(train.actual_up.mean()>=.5)),'Training prevalence':np.full(len(test),train.actual_up.mean())}
print(pd.DataFrame([{'method':name,**core.scores(y,p)} for name,p in baselines.items()]).to_string(index=False))

                method  accuracy  balanced_accuracy    brier    n
             Always up  0.460741           0.500000 0.539259 2509
Previous-day direction  0.485054           0.481608 0.514946 2509
     Training majority  0.539259           0.500000 0.460741 2509
   Training prevalence  0.539259           0.500000 0.248602 2509


## 5. Model 1: pretrained sentiment + logistic regression
The primary model uses daily count/presence, mean bullish-minus-bearish sentiment, disagreement, bullish/bearish probabilities and coin identity. Frozen **FinTwitBERT** scores text; **logistic regression** learns the market-direction target. The encoder is not fine-tuned.

Fit C candidates 0.01, 0.1 and 1.0 on training only, choose using validation Brier, threshold at 0.5. Compare with coin-only to test whether sentiment adds anything beyond asset identity. All methods use the same later-date rows.

In [6]:
done=subprocess.run([sys.executable,'headline_study.py','--universe','expanded'],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(done.stdout)
done.check_returncode()
report=json.loads(Path('docs/headline-report.json').read_text())
pred=pd.read_csv('artifacts/headline-predictions.csv')
print('CHECKS:',report['leakage_checks'])

TEST PERIOD {'start': '2026-04-12', 'end': '2026-10-21', 'rows': 2509, 'dates': 193}
Always up                 accuracy 46.0741% Brier 0.53926 n=2509
Previous-day direction    accuracy 48.5054% Brier 0.51495 n=2509
Training majority         accuracy 53.9259% Brier 0.46074 n=2509
Training prevalence       accuracy 53.9259% Brier 0.24860 n=2509
Sentiment + coin          accuracy 53.0889% Brier 0.24941 n=2509
Coin only                 accuracy 53.9259% Brier 0.24851 n=2509
Market only               accuracy 54.4041% Brier 0.24772 n=2509
Market + sentiment        accuracy 53.6070% Brier 0.24863 n=2509
KIND COMPARISONS (exploratory, full-panel Brier change; negative favours text kind)
events observed coin-days 16 delta {'mean': 0.0001305535788175643, 'interval95': [-0.0001789293487402769, 0.0005823252817242048], 'method': '1000 paired circular seven-observed-date block resamples; coins kept together'} Holm p 1.0
promotion observed coin-days 12 delta {'mean': -2.9249257720444738e-05, 'interv

## 6. Where Model 1 goes wrong
Show the most confident wrong forecasts, not only successes. A probability is not a guarantee; a mistaken market forecast is not evidence the underlying author was fraudulent. These examples are deliberately selected for error analysis, not used to adjust the trained model.

In [7]:
wrong=pred[(pred['Sentiment + coin']>=.5)!=pred.actual_up].copy()
wrong['confidence']=(wrong['Sentiment + coin']-.5).abs()
print(wrong.nlargest(5,'confidence')[['date','coin','actual_up','Sentiment + coin','post_count','actual_return']].to_string(index=False))
for row in wrong.nlargest(3,'confidence').to_dict('records'):
    day=pd.Timestamp(row['date'])
    relevant=scored[(scored.date==day)&scored.tickers.apply(lambda a:row['coin'] in a)]
    print('INPUT CONTEXT',row['coin'],str(day.date()),relevant.text.head(2).tolist())

                     date  coin  actual_up  Sentiment + coin  post_count  actual_return
2026-10-10 00:00:00+00:00  DOGE          1          0.367501           3       0.004450
2026-04-30 00:00:00+00:00  DOGE          1          0.370981           2       0.023350
2026-07-01 00:00:00+00:00 PENGU          1          0.386130           1       0.020958
2026-05-03 00:00:00+00:00 PENGU          1          0.389061           1       0.000506
2026-06-03 00:00:00+00:00   WIF          1          0.389519           1       0.085714
INPUT CONTEXT DOGE 2026-10-10 ['🐋 WHALE WATCH : Bitcoin slid 4.2% from $86592 to under $83000 triggering $550M in liquidations.\n\nMost altcoins took it harder. Seven day breakdown:\n\n=> $RENDER +4.4%\n=> $ADA 0.0%\n=> $BCH -3.6%\n=> $LTC -4.2%\n=> $SOL -4.6%\n=> $XRP -5.7%\n=> $DOGE -8.0%\n=> $LINK -9.6%\n=> $XLM -9.7%\n=> $ZEC -13.7%\n=> $UNI -16.3%\n\nBitcoin lost 1.5% on the week. Nine of eleven lost more. RENDER was the only green. ADA went flat. UNI and ZEC bot

## 7. Our addition: the larger later-date evaluation
No hand-selected 40 cases, no 15/25 sentiment quota. Evaluate the **entire natural test panel** and show results by coin, plus four expanding-date windows. Assets on the same date are dependent; the number of dates is not multiplied by the number of coins to claim independent evidence.

This is a larger version of the same archive, not an independent external dataset. Sparse text coverage remains a limitation. Missing archive posts do not prove the coin was not discussed.

In [8]:
print('TEST PANEL:',report['protocol']['splits']['test'])
print('TEXT COVERAGE:',report['coverage'])
print('NATURAL SENTIMENT COUNTS:',report['tweet_sentiment_counts'])
print(pd.DataFrame([{'coin':coin,**values} for coin,values in report['metrics']['Sentiment + coin']['per_coin'].items()]).to_string(index=False))
for window in report['walk_forward']['windows']:
    print(window['start'],window['end'],'n=',window['n'],{name:row['accuracy'] for name,row in window['metrics'].items()})
print('Expanding-window checks reuse these final dates; not independent replications.')

TEST PANEL: {'start': '2026-04-12', 'end': '2026-10-21', 'rows': 2509, 'dates': 193}
TEXT COVERAGE: {'market_bar_rows': 11432, 'all_feature_coin_days': 11202, 'eligible_original_posts': 1773, 'test_coin_days_with_posts': 219, 'test_coin_days_total': 2509}
NATURAL SENTIMENT COUNTS: {'bullish': 1238, 'bearish': 358, 'neutral': 177}
 coin  accuracy  balanced_accuracy    brier   n
 DOGE  0.461140           0.451022 0.253834 193
 SHIB  0.559585           0.524092 0.248184 193
 PEPE  0.538860           0.529282 0.249033 193
 BONK  0.580311           0.514063 0.247250 193
FLOKI  0.518135           0.493159 0.250757 193
  WIF  0.523316           0.492395 0.250137 193
TRUMP  0.544041           0.500920 0.249365 193
NEIRO  0.538860           0.496158 0.248755 193
PENGU  0.476684           0.449407 0.252805 193
 PNUT  0.606218           0.506494 0.242984 193
TURBO  0.544041           0.495283 0.247986 193
 BOME  0.487047           0.484536 0.251358 193
 MEME  0.523316           0.495691 0.249940 

## 8. One measured improvement: market context and kinds of posts
Compare sentiment alone, market history alone, and the combination. Then ask whether **events, promotion, technical/price action, or other** posts contribute differently.

Keyword buckets are exclusive per post in that priority order, approximate and not human-validated. Coin-day slices can overlap. Each kind's count and sentiment are added to market history, evaluated on all test rows. Report support, paired date-block intervals and Holm-adjusted block-sign tests. Negative Brier change favours adding the kind. A high score from a tiny slice is not a strategy.

In [9]:
from headline_study import PATTERNS
print('Fixed keyword rules:',json.dumps(PATTERNS,indent=2))
rows=[]
for item in report['kinds']:
    ci=item['paired_brier_vs_market']
    rows.append({'kind':item['kind'],'coin_days_with_kind':item['test_coin_days_with_kind'],'post_coin_incidences':item['test_posts'],'Brier_change':ci['mean'],'CI_low':ci['interval95'][0],'CI_high':ci['interval95'][1],'Holm_p':item['holm_adjusted_p']})
print(pd.DataFrame(rows).to_string(index=False))
print('Market + sentiment vs market-only paired Brier:',report['paired_accuracy_differences']['sentiment_increment_brier'])

Fixed keyword rules: {
  "events": "\\b(?:etf|approv(?:al|ed)|list(?:ing|ed)|delist|partnership|launch|upgrade|lawsuit|sec|regulat\\w*|hack(?:ed)?|exploit|burn(?:ed|ing)?|airdrop|adoption)\\b",
  "promotion": "\\b(?:buy now|last chance|don.t miss|guaranteed|moon|fomo|ape in|send it|gem|100x|1000x|lambo)\\b|\ud83d\ude80",
  "technical": "\\b(?:support|resistance|breakout|breakdown|trend(?:line)?|rsi|moving average|sma|ema|chart|consolidat\\w*|candle|volume|bullish|bearish|longs?|shorts?)\\b"
}
     kind  coin_days_with_kind  post_coin_incidences  Brier_change    CI_low  CI_high   Holm_p
   events                   16                    16      0.000131 -0.000179 0.000582 1.000000
promotion                   12                    13     -0.000029 -0.000326 0.000246 1.000000
technical                   53                    57      0.000124 -0.000052 0.000309 0.586207
    other                  161                   185      0.000587 -0.000003 0.001205 0.395802
Market + sentiment vs marke

## 9. The demo: inspect a historical forecast and reveal the market result
This uses the saved logistic model parameters, reconstructed from safe JSON—not a fabricated score. It is a historical forecast from prior information, not a live trade signal or a predictor of an individual tweet's causal effect. Change coin/date to another held-out row. Sentiment probabilities are not independently calibrated return probabilities.

In [10]:
saved=json.loads(Path('artifacts/headline-model.json').read_text())
def predict_saved(row):
    numeric=(row[saved['features']].to_numpy(dtype=float)-np.array(saved['numeric_mean']))/np.array(saved['numeric_scale'])
    categories=np.array([float(row['coin']==name.removeprefix('coin__coin_')) for name in saved['feature_names'][len(numeric):]])
    logit=float(np.r_[numeric,categories]@np.array(saved['coefficient'])[0]+saved['intercept'][0])
    return float(1/(1+np.exp(-logit)))
def replay(coin,date,reveal=False):
    day=pd.Timestamp(date,tz='UTC')
    row=test[(test.coin==coin)&(test.date==day)]
    if len(row)!=1:raise ValueError('Choose a supported coin and held-out UTC date.')
    row=row.iloc[0];prob=predict_saved(row)
    checked=pred[(pred.coin==coin)&(pd.to_datetime(pred.date,utc=True)==day)].iloc[0]
    assert np.isclose(prob,checked['Sentiment + coin'],atol=1e-12)
    context=scored[(scored.date==day)&scored.tickers.apply(lambda a:coin in a)]
    print(coin,date,'P(up), uncalibrated:',round(prob,4),'direction:','up' if prob>=.5 else 'not up')
    print('Eligible context:',context.text.head(3).tolist() or 'No eligible archive posts; not proof of silence.')
    if reveal:print('ACTUAL:',bool(row.actual_up),'return:',float(row.actual_return))
    else:print('Outcome hidden. Set reveal=True to inspect the real market result.')
first=pred[pred.post_count>0].sort_values(['date','coin']).iloc[0]
replay(first['coin'],str(first['date'])[:10],reveal=True)
print('Saved-model reconstruction verified against recorded prediction.')

PEPE 2026-04-21 P(up), uncalibrated: 0.5393 direction: up
Eligible context: ['The last final infinity stone to my bags $PEACE \n\nStrong narrative especially w/ the current war in the Middle East\n\n Matt Furie created PEACEFROG before $PEPE\n\nDouble narrative —> peace with the war & created before PEPE\n\nThis will be a core holding of mine moving forward\n✌️ 🐸']
ACTUAL: True return: 0.013297872340425565
Saved-model reconstruction verified against recorded prediction.


## 10. Results summary — model and baselines together
Use the full test-period comparison, not the most flattering coin, sentiment class or confidence band. Paired uncertainty groups coins by date. Both training-label permutation and feature-placebo controls actually refit the classifier; their accuracy nulls need not equal 50% under class imbalance.

In [11]:
order=['Always up','Previous-day direction','Training majority','Training prevalence','Sentiment + coin','Coin only','Market only','Market + sentiment']
print(pd.DataFrame([{'method':name,**{k:v for k,v in report['metrics'][name].items() if k in ['accuracy','balanced_accuracy','brier','n','n_dates']}} for name in order]).to_string(index=False))
print('Paired model-minus-baseline accuracy differences:',json.dumps(report['paired_accuracy_differences'],indent=2))
print('Controls:',report['controls'])
ci=report['paired_accuracy_differences']['Market only']['interval95']
print('Interpretation:', 'positive exploratory accuracy difference; needs future confirmation' if ci[0]>0 else ('model underperforms market-only here' if ci[1]<0 else 'no clear accuracy improvement over market-only demonstrated'))

                method  accuracy  balanced_accuracy    brier    n  n_dates
             Always up  0.460741           0.500000 0.539259 2509      193
Previous-day direction  0.485054           0.481608 0.514946 2509      193
     Training majority  0.539259           0.500000 0.460741 2509      193
   Training prevalence  0.539259           0.500000 0.248602 2509      193
      Sentiment + coin  0.530889           0.495955 0.249414 2509      193
             Coin only  0.539259           0.500000 0.248514 2509      193
           Market only  0.544041           0.513818 0.247719 2509      193
    Market + sentiment  0.536070           0.505923 0.248627 2509      193
Paired model-minus-baseline accuracy differences: {
  "Always up": {
    "mean": 0.07014746911119968,
    "interval95": [
      -0.016749701076125948,
      0.16023316062176163
    ],
    "method": "1000 paired circular seven-observed-date block resamples; coins kept together"
  },
  "Previous-day direction": {
    "mean": 

## 11. What this would and would not justify an investor doing
**Justified:** question claims that bullish posts reliably predict tomorrow; compare them with simple rules, uncertainty and actual outcomes.

**Not justified:** buying, selling, leverage, shorting, profit expectations, or saying posts caused returns. This result also does not prove all prediction impossible. No fees, slippage, execution or user-outcome study was performed.

The archive is sparse and selected, not a random sample of all X. Symbol collisions and retrospectively assigned ticker metadata may misattribute a post; this is not contract-address-verified pump.fun coverage. Current exchange listings create selection/survivorship limits. Later-listed coins have shorter training histories. Pretraining overlap, text edits and collection timing are not fully auditable. English sentiment truncates at 128 tokens. The period has been repeatedly explored; wider coverage is not independent confirmation.

*This project is a prototype and is not financial advice.*

## 12. Protocol tests and reproducibility

In [12]:
for filename in ['test_headline_study.py','test_expanded_inputs.py']:
    checked=subprocess.run([sys.executable,filename],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    print(checked.stdout);checked.check_returncode()
print('Protocol tests passed. Runtime evidence above distinguishes actual Colab from server execution.')

......
----------------------------------------------------------------------
Ran 6 tests in 0.042s

OK

.....
----------------------------------------------------------------------
Ran 5 tests in 0.056s

OK

Protocol tests passed. Runtime evidence above distinguishes actual Colab from server execution.


## Sources and credits
- Public original posts: https://huggingface.co/datasets/StephanAkkerman/financial-tweets-crypto ; dataset revision and SHA are in pipeline.py. Third-party post terms remain their own.
- Frozen sentiment: https://huggingface.co/StephanAkkerman/FinTwitBERT-sentiment ; pinned revision in pipeline.py, original work by Stephan Akkerman and Tim Koornstra.
- Actual Binance spot daily OHLCV bars, exact verified panel downloaded above. Field definitions: https://github.com/binance/binance-spot-api-docs/blob/master/rest-api.md#klinecandlestick-data . Market data is not relicensed as our code.
- Original source and older independent hourly/BTC research remain credited in the repository; expanded primary models use daily bars, not invented hourly inputs.
- PyTorch, Transformers, scikit-learn, pandas, NumPy, PyArrow, Requests and dependencies.
- Hermes Agent / OpenAI coding assistance for research, implementation, review and testing. The team must understand and explain every part.
- Veritas AI official sample structure and Finance brief §4.2.4. We changed market/source and added time-aware evaluation, broader memecoins, text-kind checks, controls and a historical replay.